In [1]:
using Revise
includet("../../scripts/single_influx.jl")

In [2]:
using ProgressMeter
using ColorSchemes
using UnPack

In [3]:
includet("../../scripts/figures_util.jl")

using GLMakie
using CairoMakie
CairoMakie.activate!()

# Prepping data for PDE run v1
This does ~10 values for li (overkill tbf...) and for each it does 5 values of K accross the unstable region. For each of those it does the first 20 of the randomly generated, solved systems

In [165]:
f = jldopen("../../cluster_env/runs/single_influx/gd5_forfigures_260212_153215.jld2");
Ks, lis, results = make_Kli_matrix_raw(f);

dominant_outcomes = map(results) do d
    mode([r.linstab_outcome for r in d])
end;

length(Ks), length(lis), size(results)

Progress: 100%|█████████████████████████████████████████| Time: 0:00:30


(50, 30, (50, 30))

In [186]:
lis_to_sample = 20:length(lis)
lis_to_sample = 12:6:length(lis)
lis_to_sample = [length(lis)]
display([lis[x] for x in lis_to_sample])
Kis_to_sample = []

for i in lis_to_sample
    first_unstable = nothing
    last_unstable = nothing
    for j in 1:length(Ks)
        if isnothing(first_unstable) && (dominant_outcomes[j, i] == :unstable)
            first_unstable = j
        end
        if isnothing(last_unstable) && (dominant_outcomes[j, i] == :stable)
            last_unstable = j - 1
        end
    end

    step = round(Int, (last_unstable - first_unstable) / 4)
    push!(Kis_to_sample, [first_unstable, first_unstable + step, first_unstable + 2 * step, first_unstable + 3 * step, last_unstable])

#=     if isnothing(first_unstable)
        push!(Kis_to_sample, [])
    else
        diff = last_unstable - first_unstable
        if diff <= 3
            push!(Kis_to_sample, [round(Int, (first_unstable + last_unstable) / 2)])
        elseif diff <= 6
            step = round(Int, (last_unstable - first_unstable) / 2)
            push!(Kis_to_sample, [first_unstable, first_unstable + step, last_unstable])
        else
            push!(Kis_to_sample, nothing)
        end
    end =#
end

1-element Vector{Float64}:
 0.999

In [187]:
systems_to_run = []

for mi in 1:length(lis_to_sample)
    for K_i in Kis_to_sample[mi]
        li_i = lis_to_sample[mi]
        
        # systems = results[K_i, li_i]
        systems = results[K_i, li_i][1:20] # only do 20 at every K, li
        
        push!(systems_to_run, (;
            K=Ks[K_i], li=lis[li_i],
            systems
        ))
    end
end

total_runs = sum(systems_to_run) do x length(x.systems) end
@show total_runs;

total_runs = 100


In [207]:
map(systems_to_run) do x
    (x.K, x.li)
end

5-element Vector{Tuple{Float64, Float64}}:
 (9.319395762340777, 0.999)
 (26.826957952797258, 0.999)
 (77.22449945836259, 0.999)
 (222.29964825261956, 0.999)
 (517.9474679231213, 0.999)

In [195]:
xx = systems_to_run[1].systems[1].params

BSMMiCRMParams{Nothing, Nothing, Nothing, Float64}(BMMiCRMParams{Nothing, Float64}([1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [0.9970490348844858, 1.0012192753894646, 1.0016973164716994, 1.0032427559667072, 1.0007575503938206, 0.9974499335963202, 1.0016615357763325, 1.0013822756596937, 1.0011867545858102, 0.9986681878723724, 1.0008076933871848, 1.0022622840887, 1.0010657405122734, 1.0011603012246617, 0.9999257215504965, 0.9996870856316563, 0.9991840022854401, 1.0017688675134553, 1.0005496357184072, 0.9999394720860076], [0.0, 0.0, 0.0, 0.0, 0.0, 9.319395762340777, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.9968707

In [204]:
rsg = get_si_sampler_for_paper(10, 1.)

JansSampler3(20, 20, LogNormal{Float64}(μ=0.0, σ=0.002302585092994046), Dirac{Float64}(value=1.0), Dirac{Int64}(value=1), Dirac{Int64}(value=10), 0.15, 1.0, Binomial{Float64}(n=20, p=0.15), LogNormal{Float64}(μ=0.0, σ=0.002302585092994046), Dirac{Float64}(value=1.0), LogNormal{Float64}(μ=0.0, σ=0.002302585092994046), Dirac{Float64}(value=1.0), Dirac{Float64}(value=0.0), Dirac{Float64}(value=1.0), Dirac{Float64}(value=1.0), nothing)

In [214]:
xx = []
for i in 1:10
    for _ in 1:5
        push!(xx, i)
    end
end
xx

50-element Vector{Any}:
  1
  1
  1
  1
  1
  2
  2
  2
  2
  2
  3
  3
  3
  ⋮
  8
  8
  9
  9
  9
  9
  9
 10
 10
 10
 10
 10

In [217]:
yy = reshape(xx, 5, 10)

5×10 Matrix{Any}:
 1  2  3  4  5  6  7  8  9  10
 1  2  3  4  5  6  7  8  9  10
 1  2  3  4  5  6  7  8  9  10
 1  2  3  4  5  6  7  8  9  10
 1  2  3  4  5  6  7  8  9  10

In [219]:
yy[:,1]

5-element Vector{Any}:
 1
 1
 1
 1
 1

In [188]:
jldsave("../../cluster_env/runs/single_influx_pdes2/systems3_just_l1.jld2"; systems_to_run)